In [2]:
import pandas as pd
import numpy as np
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker 
from xgboost import XGBClassifier
from sklearn.inspection import partial_dependence
import random
import os
from itertools import product


from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OrdinalEncoder
from sksurv.preprocessing import OneHotEncoder
from sklearn_pandas import DataFrameMapper

import torch
import torch.nn as nn
import torchtuples as tt

from torchtuples.callbacks import EarlyStopping
from sklearn.neural_network import MLPClassifier
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import log_loss
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import KFold

In [3]:
df_train = pd.read_csv('df_train.csv')
df_test = pd.read_csv('df_test.csv')

In [4]:
df_train, df_val = train_test_split(df_train, test_size=0.2, random_state=42)

In [5]:
# det dummy
dummy_f = ['REGION','MARST', 'RACEA', 'USBORN', 'HSTATYR', 'ALCSTAT1', 'SMOKESTATUS2']
df_train = pd.get_dummies(df_train, columns=dummy_f, prefix=dummy_f).astype(int)
df_val = pd.get_dummies(df_val, columns=dummy_f, prefix=dummy_f).astype(int)
df_test = pd.get_dummies(df_test, columns=dummy_f, prefix=dummy_f).astype(int)

In [6]:
# list of columns to standardize
cols_stand = ['AGE', 'BMICALC', 'HRSLEEP']
              
cols_norm = ['FAMSIZE', 'EDUCREC1', 'POVERTY', 'FSRAWSCORE', 'ALC5UPYR', 'ALCAMT', 'ALCDAYSYR',
              'CIGSDAY', 'MOD10DMIN', 'MOD10FWK', 'VIG10DMIN', 'VIG10FWK', 'STRONGFWK', 'AEFFORT', 
              'AFEELINT1MO', 'AWORTHLESS', 'WRYMEDCST', 'WRYRET',
              'SEX', 'HINOTCOVE', 'HIPRIVATEE', 'HIMCAIDE', 'HIMCAREE', 'ARTHGLUPEV',
              'BLIND', 'CANCEREV', 'CHEARTDIEV', 'CPOXEV', 'DIABETICEV', 'EMPHYSEMEV',
              'HEARTATTEV', 'HEARTCONEV', 'HYPERTENEV', 'KIDNEYWKYR', 'LIVERCHRON',
              'LIVERCONYR', 'STROKEV']

# list of columns not to standardize
cols_leave = [col for col in df_train.columns if col not in cols_stand + cols_norm + ['TIMETOEVENT', 'MORTSTAT']]

# create mapper entries
standardize = [([col], StandardScaler()) for col in cols_stand]
normalize = [([col], MinMaxScaler()) for col in cols_norm]
leave = [(col, None) for col in cols_leave]

# combine into one mapper
x_mapper = DataFrameMapper(standardize + normalize + leave, df_out=False)

# prepare X matrices
x_train = df_train.drop(columns=['MORTSTAT', 'TIMETOEVENT'])
x_val = df_val.drop(columns=['MORTSTAT', 'TIMETOEVENT'])
x_test = df_test.drop(columns=['MORTSTAT', 'TIMETOEVENT'])

In [7]:
# transform with DataFrameMapper and convert to PyTorch tensors
x_train = torch.tensor(x_mapper.fit_transform(df_train.drop(columns=['MORTSTAT', 'TIMETOEVENT'])), dtype=torch.float32)
x_val = torch.tensor(x_mapper.transform(df_val.drop(columns=['MORTSTAT', 'TIMETOEVENT'])), dtype=torch.float32)
x_test = torch.tensor(x_mapper.transform(df_test.drop(columns=['MORTSTAT', 'TIMETOEVENT'])), dtype=torch.float32)

In [8]:
# define three-year mortality status as PyTorch tensors
y_train = torch.tensor(np.where((df_train["MORTSTAT"] == 1) & (df_train["TIMETOEVENT"] <= 3), 1, 0), dtype=torch.float32).view(-1, 1)
y_val = torch.tensor(np.where((df_val["MORTSTAT"] == 1) & (df_val["TIMETOEVENT"] <= 3), 1, 0), dtype=torch.float32).view(-1, 1)
y_test = torch.tensor(np.where((df_test["MORTSTAT"] == 1) & (df_test["TIMETOEVENT"] <= 3), 1, 0), dtype=torch.float32).view(-1, 1)

In [9]:
# define loss funtions
def ll3y(estimator, X, y_df):
    try:
        death_p = torch.sigmoid(estimator.predict(X))

        # check for NaNs in predictions
        if torch.isnan(death_p).any():
            print("NaNs in prediction — assigning score = -100")
            return -100.0
        
        # otherwise return log_loss
        return -log_loss(y_df, death_p)
    
    except Exception as e:
        print(f"Error during scoring: {e} — assigning score = -100")
        return -100.0
    

def bs3y(estimator, X, y_df):
    try:
        death_p = torch.sigmoid(estimator.predict(X))

        # check for NaNs in predictions
        if torch.isnan(death_p).any():
            print("NaNs in prediction — assigning score = -100")
            return -100.0
        
        # otherwise return log_loss
        return -brier_score_loss(y_df, death_p)
    
    except Exception as e:
        print(f"Error during scoring: {e} — assigning score = -100")
        return -100.0

In [10]:
# GridSearchCV

In [11]:
# set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ['PYTHONHASHSEED'] = str(SEED)


# define parameter grid
param_grid = {'batch_size': [200, 400, 800],
              'epochs': [30, 60], 
              'lr': [0.001,0.005, 0.01],
              'dropout': [0.0, 0.2, 0.5],
              'num_nodes': [[32, 16], [32, 32], [64, 32], [32], [16]]}

param_combos = list(product(param_grid['batch_size'], param_grid['epochs'], param_grid['lr'], param_grid['dropout'], param_grid['num_nodes']))



# compute cross-validation loss for each combination of hyperparatmers
n_splits = 4
results_ll = []
results_bs = []

for i, (batch_size, epochs, lr, dropout, num_nodes) in enumerate(param_combos):
    print(f"\n=== Grid {i+1}/{len(param_combos)}:  batch_size={batch_size},  epochs={epochs}, lr={lr}, dropout={dropout}, nodes={num_nodes} ===")

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    scores_ll = []
    scores_bs = []

    for fold, (train_idx, tst_idx) in enumerate(kf.split(x_train)):
        print(f"  Fold {fold + 1}/{n_splits}")
        
        x_tr, x_tst = x_train[train_idx], x_train[tst_idx]
        y_tr, y_tst = y_train[train_idx], y_train[tst_idx]
        
        # define architecure
        in_features = x_tr.shape[1]
        net = tt.practical.MLPVanilla(in_features, num_nodes, 1, activation=nn.ReLU, dropout=dropout)
        # define model
        loss_fn = nn.BCEWithLogitsLoss()
        model = tt.Model(net, loss_fn, tt.optim.Adam)  
        model.optimizer.set_lr(lr) 
        # for early stopping
        checkpoint_path = 'local_checkpoints0.pt'
        if os.path.exists(checkpoint_path):
            os.remove(checkpoint_path)
        callbacks = [EarlyStopping(patience=20, file_path=checkpoint_path)]
        
        # fit model
        model.fit(x_tr, y_tr, batch_size=batch_size, epochs=epochs, val_data=(x_val, y_val), shuffle=True, verbose=False, callbacks=callbacks)
        
        # compute loss
        score_ll = ll3y(model, x_tst, y_tst)
        scores_ll.append(score_ll)
        score_bs = bs3y(model, x_tst, y_tst)
        scores_bs.append(score_bs)
        print(f"    LL is: {-score_ll:.4f}")
        print(f"    BS is: {-score_bs:.4f}")
    # save mean loss vales
    mean_score_ll = np.mean(scores_ll)
    std_score_ll = np.std(scores_ll)
    mean_score_bs = np.mean(scores_bs)
    std_score_bs = np.std(scores_bs)
    results_ll.append({'batch_size': batch_size, 'epochs': epochs, 'lr': lr, 'dropout': dropout,
                    'num_nodes': str(num_nodes), 'mean_neg_logloss': mean_score_ll, 'std_neg_logloss': std_score_ll})
    results_bs.append({'batch_size': batch_size, 'epochs': epochs, 'lr': lr, 'dropout': dropout,
                    'num_nodes': str(num_nodes), 'mean_neg_logloss': mean_score_bs, 'std_neg_logloss': std_score_bs})


=== Grid 1/270:  batch_size=200,  epochs=30, lr=0.001, dropout=0.0, nodes=[32, 16] ===
  Fold 1/4
    LL is: 0.1169
    BS is: 0.0303
  Fold 2/4
    LL is: 0.1154
    BS is: 0.0304
  Fold 3/4
    LL is: 0.1274
    BS is: 0.0333
  Fold 4/4
    LL is: 0.1117
    BS is: 0.0290

=== Grid 2/270:  batch_size=200,  epochs=30, lr=0.001, dropout=0.0, nodes=[32, 32] ===
  Fold 1/4
    LL is: 0.1170
    BS is: 0.0308
  Fold 2/4
    LL is: 0.1172
    BS is: 0.0308
  Fold 3/4
    LL is: 0.1240
    BS is: 0.0325
  Fold 4/4
    LL is: 0.1110
    BS is: 0.0289

=== Grid 3/270:  batch_size=200,  epochs=30, lr=0.001, dropout=0.0, nodes=[64, 32] ===
  Fold 1/4
    LL is: 0.1214
    BS is: 0.0307
  Fold 2/4
    LL is: 0.1185
    BS is: 0.0311
  Fold 3/4
    LL is: 0.1252
    BS is: 0.0327
  Fold 4/4
    LL is: 0.1127
    BS is: 0.0290

=== Grid 4/270:  batch_size=200,  epochs=30, lr=0.001, dropout=0.0, nodes=[32] ===
  Fold 1/4
    LL is: 0.1162
    BS is: 0.0306
  Fold 2/4
    LL is: 0.1129
    BS is: 0

    LL is: 0.1094
    BS is: 0.0288

=== Grid 31/270:  batch_size=200,  epochs=30, lr=0.01, dropout=0.0, nodes=[32, 16] ===
  Fold 1/4
    LL is: 0.1189
    BS is: 0.0308
  Fold 2/4
    LL is: 0.1146
    BS is: 0.0304
  Fold 3/4
    LL is: 0.1213
    BS is: 0.0322
  Fold 4/4
    LL is: 0.1092
    BS is: 0.0285

=== Grid 32/270:  batch_size=200,  epochs=30, lr=0.01, dropout=0.0, nodes=[32, 32] ===
  Fold 1/4
    LL is: 0.1143
    BS is: 0.0301
  Fold 2/4
    LL is: 0.1121
    BS is: 0.0299
  Fold 3/4
    LL is: 0.1212
    BS is: 0.0320
  Fold 4/4
    LL is: 0.1122
    BS is: 0.0292

=== Grid 33/270:  batch_size=200,  epochs=30, lr=0.01, dropout=0.0, nodes=[64, 32] ===
  Fold 1/4
    LL is: 0.1157
    BS is: 0.0304
  Fold 2/4
    LL is: 0.1165
    BS is: 0.0309
  Fold 3/4
    LL is: 0.1219
    BS is: 0.0320
  Fold 4/4
    LL is: 0.1092
    BS is: 0.0284

=== Grid 34/270:  batch_size=200,  epochs=30, lr=0.01, dropout=0.0, nodes=[32] ===
  Fold 1/4
    LL is: 0.1152
    BS is: 0.0302
  Fol

    LL is: 0.1217
    BS is: 0.0321
  Fold 4/4
    LL is: 0.1085
    BS is: 0.0284

=== Grid 61/270:  batch_size=200,  epochs=60, lr=0.005, dropout=0.0, nodes=[32, 16] ===
  Fold 1/4
    LL is: 0.1167
    BS is: 0.0302
  Fold 2/4
    LL is: 0.1142
    BS is: 0.0303
  Fold 3/4
    LL is: 0.1219
    BS is: 0.0323
  Fold 4/4
    LL is: 0.1120
    BS is: 0.0292

=== Grid 62/270:  batch_size=200,  epochs=60, lr=0.005, dropout=0.0, nodes=[32, 32] ===
  Fold 1/4
    LL is: 0.1156
    BS is: 0.0302
  Fold 2/4
    LL is: 0.1145
    BS is: 0.0306
  Fold 3/4
    LL is: 0.1243
    BS is: 0.0326
  Fold 4/4
    LL is: 0.1133
    BS is: 0.0294

=== Grid 63/270:  batch_size=200,  epochs=60, lr=0.005, dropout=0.0, nodes=[64, 32] ===
  Fold 1/4
    LL is: 0.1153
    BS is: 0.0303
  Fold 2/4
    LL is: 0.1136
    BS is: 0.0302
  Fold 3/4
    LL is: 0.1217
    BS is: 0.0318
  Fold 4/4
    LL is: 0.1147
    BS is: 0.0294

=== Grid 64/270:  batch_size=200,  epochs=60, lr=0.005, dropout=0.0, nodes=[32] ===
 

    LL is: 0.1124
    BS is: 0.0300
  Fold 3/4
    LL is: 0.1213
    BS is: 0.0322
  Fold 4/4
    LL is: 0.1090
    BS is: 0.0286

=== Grid 91/270:  batch_size=400,  epochs=30, lr=0.001, dropout=0.0, nodes=[32, 16] ===
  Fold 1/4
    LL is: 0.1174
    BS is: 0.0305
  Fold 2/4
    LL is: 0.1160
    BS is: 0.0305
  Fold 3/4
    LL is: 0.1242
    BS is: 0.0326
  Fold 4/4
    LL is: 0.1132
    BS is: 0.0291

=== Grid 92/270:  batch_size=400,  epochs=30, lr=0.001, dropout=0.0, nodes=[32, 32] ===
  Fold 1/4
    LL is: 0.1184
    BS is: 0.0307
  Fold 2/4
    LL is: 0.1176
    BS is: 0.0308
  Fold 3/4
    LL is: 0.1254
    BS is: 0.0331
  Fold 4/4
    LL is: 0.1134
    BS is: 0.0295

=== Grid 93/270:  batch_size=400,  epochs=30, lr=0.001, dropout=0.0, nodes=[64, 32] ===
  Fold 1/4
    LL is: 0.1230
    BS is: 0.0322
  Fold 2/4
    LL is: 0.1164
    BS is: 0.0311
  Fold 3/4
    LL is: 0.1269
    BS is: 0.0330
  Fold 4/4
    LL is: 0.1145
    BS is: 0.0298

=== Grid 94/270:  batch_size=400,  epo

    LL is: 0.1130
    BS is: 0.0298
  Fold 2/4
    LL is: 0.1122
    BS is: 0.0300
  Fold 3/4
    LL is: 0.1207
    BS is: 0.0318
  Fold 4/4
    LL is: 0.1095
    BS is: 0.0287

=== Grid 121/270:  batch_size=400,  epochs=30, lr=0.01, dropout=0.0, nodes=[32, 16] ===
  Fold 1/4
    LL is: 0.1152
    BS is: 0.0303
  Fold 2/4
    LL is: 0.1140
    BS is: 0.0302
  Fold 3/4
    LL is: 0.1232
    BS is: 0.0323
  Fold 4/4
    LL is: 0.1132
    BS is: 0.0297

=== Grid 122/270:  batch_size=400,  epochs=30, lr=0.01, dropout=0.0, nodes=[32, 32] ===
  Fold 1/4
    LL is: 0.1143
    BS is: 0.0301
  Fold 2/4
    LL is: 0.1151
    BS is: 0.0305
  Fold 3/4
    LL is: 0.1242
    BS is: 0.0323
  Fold 4/4
    LL is: 0.1117
    BS is: 0.0291

=== Grid 123/270:  batch_size=400,  epochs=30, lr=0.01, dropout=0.0, nodes=[64, 32] ===
  Fold 1/4
    LL is: 0.1146
    BS is: 0.0302
  Fold 2/4
    LL is: 0.1155
    BS is: 0.0306
  Fold 3/4
    LL is: 0.1247
    BS is: 0.0325
  Fold 4/4
    LL is: 0.1111
    BS is:

    LL is: 0.1132
    BS is: 0.0299
  Fold 2/4
    LL is: 0.1128
    BS is: 0.0300
  Fold 3/4
    LL is: 0.1205
    BS is: 0.0318
  Fold 4/4
    LL is: 0.1092
    BS is: 0.0286

=== Grid 151/270:  batch_size=400,  epochs=60, lr=0.005, dropout=0.0, nodes=[32, 16] ===
  Fold 1/4
    LL is: 0.1153
    BS is: 0.0303
  Fold 2/4
    LL is: 0.1186
    BS is: 0.0307
  Fold 3/4
    LL is: 0.1231
    BS is: 0.0322
  Fold 4/4
    LL is: 0.1151
    BS is: 0.0292

=== Grid 152/270:  batch_size=400,  epochs=60, lr=0.005, dropout=0.0, nodes=[32, 32] ===
  Fold 1/4
    LL is: 0.1166
    BS is: 0.0304
  Fold 2/4
    LL is: 0.1157
    BS is: 0.0307
  Fold 3/4
    LL is: 0.1255
    BS is: 0.0330
  Fold 4/4
    LL is: 0.1125
    BS is: 0.0293

=== Grid 153/270:  batch_size=400,  epochs=60, lr=0.005, dropout=0.0, nodes=[64, 32] ===
  Fold 1/4
    LL is: 0.1194
    BS is: 0.0312
  Fold 2/4
    LL is: 0.1144
    BS is: 0.0303
  Fold 3/4
    LL is: 0.1229
    BS is: 0.0323
  Fold 4/4
    LL is: 0.1123
    BS 

    LL is: 0.1128
    BS is: 0.0298
  Fold 2/4
    LL is: 0.1120
    BS is: 0.0299
  Fold 3/4
    LL is: 0.1204
    BS is: 0.0320
  Fold 4/4
    LL is: 0.1096
    BS is: 0.0286

=== Grid 181/270:  batch_size=800,  epochs=30, lr=0.001, dropout=0.0, nodes=[32, 16] ===
  Fold 1/4
    LL is: 0.1202
    BS is: 0.0309
  Fold 2/4
    LL is: 0.1196
    BS is: 0.0314
  Fold 3/4
    LL is: 0.1235
    BS is: 0.0324
  Fold 4/4
    LL is: 0.1140
    BS is: 0.0294

=== Grid 182/270:  batch_size=800,  epochs=30, lr=0.001, dropout=0.0, nodes=[32, 32] ===
  Fold 1/4
    LL is: 0.1187
    BS is: 0.0312
  Fold 2/4
    LL is: 0.1168
    BS is: 0.0307
  Fold 3/4
    LL is: 0.1239
    BS is: 0.0323
  Fold 4/4
    LL is: 0.1137
    BS is: 0.0295

=== Grid 183/270:  batch_size=800,  epochs=30, lr=0.001, dropout=0.0, nodes=[64, 32] ===
  Fold 1/4
    LL is: 0.1199
    BS is: 0.0312
  Fold 2/4
    LL is: 0.1191
    BS is: 0.0314
  Fold 3/4
    LL is: 0.1255
    BS is: 0.0329
  Fold 4/4
    LL is: 0.1149
    BS 

    LL is: 0.1133
    BS is: 0.0298
  Fold 2/4
    LL is: 0.1119
    BS is: 0.0298
  Fold 3/4
    LL is: 0.1203
    BS is: 0.0319
  Fold 4/4
    LL is: 0.1087
    BS is: 0.0283

=== Grid 211/270:  batch_size=800,  epochs=30, lr=0.01, dropout=0.0, nodes=[32, 16] ===
  Fold 1/4
    LL is: 0.1152
    BS is: 0.0301
  Fold 2/4
    LL is: 0.1144
    BS is: 0.0305
  Fold 3/4
    LL is: 0.1229
    BS is: 0.0321
  Fold 4/4
    LL is: 0.1116
    BS is: 0.0290

=== Grid 212/270:  batch_size=800,  epochs=30, lr=0.01, dropout=0.0, nodes=[32, 32] ===
  Fold 1/4
    LL is: 0.1164
    BS is: 0.0305
  Fold 2/4
    LL is: 0.1150
    BS is: 0.0306
  Fold 3/4
    LL is: 0.1212
    BS is: 0.0321
  Fold 4/4
    LL is: 0.1110
    BS is: 0.0290

=== Grid 213/270:  batch_size=800,  epochs=30, lr=0.01, dropout=0.0, nodes=[64, 32] ===
  Fold 1/4
    LL is: 0.1151
    BS is: 0.0300
  Fold 2/4
    LL is: 0.1167
    BS is: 0.0308
  Fold 3/4
    LL is: 0.1234
    BS is: 0.0321
  Fold 4/4
    LL is: 0.1127
    BS is:

    LL is: 0.1133
    BS is: 0.0298
  Fold 2/4
    LL is: 0.1128
    BS is: 0.0301
  Fold 3/4
    LL is: 0.1206
    BS is: 0.0321
  Fold 4/4
    LL is: 0.1112
    BS is: 0.0289

=== Grid 241/270:  batch_size=800,  epochs=60, lr=0.005, dropout=0.0, nodes=[32, 16] ===
  Fold 1/4
    LL is: 0.1152
    BS is: 0.0302
  Fold 2/4
    LL is: 0.1144
    BS is: 0.0304
  Fold 3/4
    LL is: 0.1221
    BS is: 0.0322
  Fold 4/4
    LL is: 0.1126
    BS is: 0.0292

=== Grid 242/270:  batch_size=800,  epochs=60, lr=0.005, dropout=0.0, nodes=[32, 32] ===
  Fold 1/4
    LL is: 0.1173
    BS is: 0.0308
  Fold 2/4
    LL is: 0.1149
    BS is: 0.0305
  Fold 3/4
    LL is: 0.1250
    BS is: 0.0327
  Fold 4/4
    LL is: 0.1123
    BS is: 0.0292

=== Grid 243/270:  batch_size=800,  epochs=60, lr=0.005, dropout=0.0, nodes=[64, 32] ===
  Fold 1/4
    LL is: 0.1201
    BS is: 0.0308
  Fold 2/4
    LL is: 0.1183
    BS is: 0.0311
  Fold 3/4
    LL is: 0.1228
    BS is: 0.0324
  Fold 4/4
    LL is: 0.1152
    BS 

    LL is: 0.1135
    BS is: 0.0299
  Fold 2/4
    LL is: 0.1116
    BS is: 0.0298
  Fold 3/4
    LL is: 0.1201
    BS is: 0.0317
  Fold 4/4
    LL is: 0.1102
    BS is: 0.0287


In [12]:
# report best result from best to worst

results_ll_df = pd.DataFrame(results_ll)
results_ll_df['ll3y'] = - results_ll_df['mean_neg_logloss']

print(results_ll_df.sort_values(by='mean_neg_logloss', ascending=False).to_string(index=False))

 batch_size  epochs    lr  dropout num_nodes  mean_neg_logloss  std_neg_logloss     ll3y
        200      60 0.010      0.5      [32]         -0.113433         0.003937 0.113433
        800      30 0.005      0.5      [16]         -0.113518         0.004251 0.113518
        200      60 0.005      0.5  [64, 32]         -0.113543         0.004399 0.113543
        400      30 0.005      0.2      [16]         -0.113645         0.004067 0.113645
        400      30 0.005      0.5      [32]         -0.113693         0.004344 0.113693
        400      60 0.010      0.5      [16]         -0.113698         0.004034 0.113698
        800      60 0.005      0.2      [16]         -0.113712         0.004363 0.113712
        200      60 0.005      0.5      [32]         -0.113740         0.004105 0.113740
        400      60 0.005      0.2      [16]         -0.113766         0.003826 0.113766
        800      30 0.010      0.2      [16]         -0.113807         0.003934 0.113807
        200      60 0

In [13]:
results_bs_df = pd.DataFrame(results_bs)
results_bs_df['bs3y'] = - results_bs_df['mean_neg_logloss']

print(results_bs_df.sort_values(by='mean_neg_logloss', ascending=False).to_string(index=False))

 batch_size  epochs    lr  dropout num_nodes  mean_neg_logloss  std_neg_logloss     bs3y
        800      60 0.005      0.2      [16]         -0.029968         0.001174 0.029968
        200      60 0.005      0.5  [64, 32]         -0.029968         0.001194 0.029968
        800      30 0.005      0.5      [16]         -0.029977         0.001280 0.029977
        400      30 0.005      0.5      [32]         -0.029997         0.001202 0.029997
        400      30 0.005      0.2      [16]         -0.030002         0.001243 0.030002
        800      30 0.010      0.5  [64, 32]         -0.030005         0.001295 0.030005
        200      30 0.005      0.2      [32]         -0.030007         0.001190 0.030007
        400      30 0.001      0.2      [16]         -0.030013         0.001199 0.030013
        400      60 0.005      0.2      [16]         -0.030031         0.001153 0.030031
        200      60 0.010      0.5      [32]         -0.030032         0.001141 0.030032
        400      30 0

In [ ]:
#______________________________________________________________________________________________________________________________

In [ ]:
#______________________________________________________________________________________________________________________________

In [ ]:
# DeepSC1L

In [14]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED) 
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ['PYTHONHASHSEED'] = str(SEED)

# define the network
in_features = x_train.shape[1]
net = tt.practical.MLPVanilla(in_features, [32], 1, activation=nn.ReLU, dropout=0.5)

# model and optimizer

loss_fn = nn.BCEWithLogitsLoss()
model = tt.Model(net, loss_fn, tt.optim.Adam) 
model.optimizer.set_lr(0.010) 

# training setup     
batch_size = 200
epochs = 60
# for early stoping
checkpoint_path = 'best_model.pt'
if os.path.exists(checkpoint_path):
    os.remove(checkpoint_path)
callbacks = [EarlyStopping(patience=20, file_path=checkpoint_path)]

# fit 
log = model.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, val_data=(x_test, y_test), shuffle=True,
                callbacks=callbacks, verbose=True)

0:	[0s / 0s],		train_loss: 0.1623,	val_loss: 0.1104
1:	[0s / 1s],		train_loss: 0.1193,	val_loss: 0.1090
2:	[0s / 1s],		train_loss: 0.1186,	val_loss: 0.1091
3:	[0s / 2s],		train_loss: 0.1167,	val_loss: 0.1090
4:	[0s / 2s],		train_loss: 0.1173,	val_loss: 0.1097
5:	[0s / 3s],		train_loss: 0.1165,	val_loss: 0.1082
6:	[0s / 4s],		train_loss: 0.1159,	val_loss: 0.1092
7:	[0s / 4s],		train_loss: 0.1156,	val_loss: 0.1082
8:	[0s / 5s],		train_loss: 0.1163,	val_loss: 0.1090
9:	[0s / 5s],		train_loss: 0.1152,	val_loss: 0.1100
10:	[0s / 6s],		train_loss: 0.1159,	val_loss: 0.1078
11:	[0s / 6s],		train_loss: 0.1153,	val_loss: 0.1091
12:	[0s / 7s],		train_loss: 0.1153,	val_loss: 0.1082
13:	[0s / 7s],		train_loss: 0.1147,	val_loss: 0.1089
14:	[0s / 8s],		train_loss: 0.1146,	val_loss: 0.1086
15:	[0s / 8s],		train_loss: 0.1152,	val_loss: 0.1085
16:	[0s / 9s],		train_loss: 0.1152,	val_loss: 0.1083
17:	[0s / 9s],		train_loss: 0.1152,	val_loss: 0.1087
18:	[0s / 10s],		train_loss: 0.1151,	val_loss: 0.1082
19

In [15]:
# get probabilites
probs_train = torch.sigmoid(model.predict(x_train))
probs_test = torch.sigmoid(model.predict(x_test))

print(f'log-loss is: {log_loss(y_train, probs_train):.4f}')
print(f'log-loss os: {log_loss(y_test, probs_test):.4f}')
print(f'BS is: {brier_score_loss(y_train, probs_train):.4f}')
print(f'BS os: {brier_score_loss(y_test, probs_test):.4f}')

log-loss is: 0.1111
log-loss os: 0.1093
BS is: 0.0295
BS os: 0.0292


In [16]:
# DeepSC2L

In [17]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)  
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ['PYTHONHASHSEED'] = str(SEED)

# define the network
in_features = x_train.shape[1]
net = tt.practical.MLPVanilla(in_features, [16], 1, activation=nn.ReLU, dropout=0.5)

# model and optimizer
loss_fn = nn.BCEWithLogitsLoss()
model = tt.Model(net, loss_fn, tt.optim.Adam)  
model.optimizer.set_lr(0.005)  

# training setup   
batch_size = 800
epochs = 30
# for early stopping
checkpoint_path = 'best_model.pt'
if os.path.exists(checkpoint_path):
    os.remove(checkpoint_path)
callbacks = [EarlyStopping(patience=20, file_path=checkpoint_path)]

# fit
log = model.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, val_data=(x_test, y_test), shuffle=True,
                callbacks=callbacks, verbose=True)

0:	[0s / 0s],		train_loss: 0.4472,	val_loss: 0.1464
1:	[0s / 0s],		train_loss: 0.1419,	val_loss: 0.1109
2:	[0s / 0s],		train_loss: 0.1270,	val_loss: 0.1097
3:	[0s / 0s],		train_loss: 0.1238,	val_loss: 0.1090
4:	[0s / 0s],		train_loss: 0.1207,	val_loss: 0.1084
5:	[0s / 1s],		train_loss: 0.1198,	val_loss: 0.1086
6:	[0s / 1s],		train_loss: 0.1202,	val_loss: 0.1082
7:	[0s / 1s],		train_loss: 0.1192,	val_loss: 0.1084
8:	[0s / 1s],		train_loss: 0.1192,	val_loss: 0.1080
9:	[0s / 1s],		train_loss: 0.1194,	val_loss: 0.1082
10:	[0s / 1s],		train_loss: 0.1183,	val_loss: 0.1080
11:	[0s / 2s],		train_loss: 0.1188,	val_loss: 0.1083
12:	[0s / 2s],		train_loss: 0.1188,	val_loss: 0.1081
13:	[0s / 2s],		train_loss: 0.1172,	val_loss: 0.1081
14:	[0s / 2s],		train_loss: 0.1175,	val_loss: 0.1081
15:	[0s / 2s],		train_loss: 0.1175,	val_loss: 0.1081
16:	[0s / 3s],		train_loss: 0.1179,	val_loss: 0.1079
17:	[0s / 3s],		train_loss: 0.1180,	val_loss: 0.1082
18:	[0s / 3s],		train_loss: 0.1180,	val_loss: 0.1082
19:

In [18]:
# get probabilites
probs_train = torch.sigmoid(model.predict(x_train))
probs_test = torch.sigmoid(model.predict(x_test))

print(f'log-loss is: {log_loss(y_train, probs_train):.4f}')
print(f'log-loss os: {log_loss(y_test, probs_test):.4f}')
print(f'BS is: {brier_score_loss(y_train, probs_train):.4f}')
print(f'BS os: {brier_score_loss(y_test, probs_test):.4f}')

log-loss is: 0.1110
log-loss os: 0.1094
BS is: 0.0294
BS os: 0.0291


In [19]:
# DeepSC3L

In [20]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)  
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ['PYTHONHASHSEED'] = str(SEED)

# define the network
in_features = x_train.shape[1]
net = tt.practical.MLPVanilla(in_features, [64,32], 1, activation=nn.ReLU, dropout=0.5)

# model and optimizer=
loss_fn = nn.BCEWithLogitsLoss()
model = tt.Model(net, loss_fn, tt.optim.Adam)  
model.optimizer.set_lr(0.005)  

# training setup 
batch_size = 200
epochs = 60
# for early stopping
checkpoint_path = 'best_model.pt'
if os.path.exists(checkpoint_path):
    os.remove(checkpoint_path)
callbacks = [EarlyStopping(patience=20, file_path=checkpoint_path)]

# fit
log = model.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, val_data=(x_test, y_test), shuffle=True,
                callbacks=callbacks, verbose=True)

0:	[0s / 0s],		train_loss: 0.2204,	val_loss: 0.1108
1:	[0s / 1s],		train_loss: 0.1249,	val_loss: 0.1099
2:	[0s / 2s],		train_loss: 0.1215,	val_loss: 0.1089
3:	[0s / 2s],		train_loss: 0.1191,	val_loss: 0.1084
4:	[0s / 3s],		train_loss: 0.1193,	val_loss: 0.1087
5:	[0s / 4s],		train_loss: 0.1181,	val_loss: 0.1088
6:	[0s / 4s],		train_loss: 0.1174,	val_loss: 0.1093
7:	[0s / 5s],		train_loss: 0.1178,	val_loss: 0.1088
8:	[0s / 6s],		train_loss: 0.1176,	val_loss: 0.1089
9:	[0s / 7s],		train_loss: 0.1171,	val_loss: 0.1078
10:	[0s / 7s],		train_loss: 0.1148,	val_loss: 0.1080
11:	[0s / 8s],		train_loss: 0.1154,	val_loss: 0.1077
12:	[0s / 9s],		train_loss: 0.1158,	val_loss: 0.1086
13:	[0s / 10s],		train_loss: 0.1152,	val_loss: 0.1081
14:	[0s / 10s],		train_loss: 0.1150,	val_loss: 0.1080
15:	[0s / 11s],		train_loss: 0.1146,	val_loss: 0.1088
16:	[0s / 12s],		train_loss: 0.1142,	val_loss: 0.1089
17:	[0s / 13s],		train_loss: 0.1140,	val_loss: 0.1085
18:	[0s / 14s],		train_loss: 0.1140,	val_loss: 0.10

In [21]:
# get probabilites
probs_train = torch.sigmoid(model.predict(x_train))
probs_test = torch.sigmoid(model.predict(x_test))

print(f'log-loss is: {log_loss(y_train, probs_train):.4f}')
print(f'log-loss os: {log_loss(y_test, probs_test):.4f}')
print(f'BS is: {brier_score_loss(y_train, probs_train):.4f}')
print(f'BS os: {brier_score_loss(y_test, probs_test):.4f}')

log-loss is: 0.1071
log-loss os: 0.1093
BS is: 0.0290
BS os: 0.0292


In [22]:
#______________________________________________________________________________________________________________________________

In [23]:
# DeepSC1B

In [24]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)  
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ['PYTHONHASHSEED'] = str(SEED)

# define the network
in_features = x_train.shape[1]
net = tt.practical.MLPVanilla(in_features, [16], 1, activation=nn.ReLU, dropout=0.2)

# model and optimizer
loss_fn = nn.BCEWithLogitsLoss()
model = tt.Model(net, loss_fn, tt.optim.Adam)  
model.optimizer.set_lr(0.005)  

# training setup  
batch_size = 800
epochs = 60
# early stopping
checkpoint_path = 'best_model.pt'
if os.path.exists(checkpoint_path):
    os.remove(checkpoint_path)
callbacks = [EarlyStopping(patience=20, file_path=checkpoint_path)]

# fit
log = model.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, val_data=(x_test, y_test), shuffle=True,
                callbacks=callbacks, verbose=True)

0:	[0s / 0s],		train_loss: 0.3983,	val_loss: 0.1280
1:	[0s / 0s],		train_loss: 0.1269,	val_loss: 0.1102
2:	[0s / 0s],		train_loss: 0.1190,	val_loss: 0.1089
3:	[0s / 0s],		train_loss: 0.1171,	val_loss: 0.1090
4:	[0s / 0s],		train_loss: 0.1152,	val_loss: 0.1081
5:	[0s / 1s],		train_loss: 0.1144,	val_loss: 0.1083
6:	[0s / 1s],		train_loss: 0.1154,	val_loss: 0.1080
7:	[0s / 1s],		train_loss: 0.1140,	val_loss: 0.1081
8:	[0s / 1s],		train_loss: 0.1136,	val_loss: 0.1078
9:	[0s / 1s],		train_loss: 0.1135,	val_loss: 0.1079
10:	[0s / 2s],		train_loss: 0.1134,	val_loss: 0.1077
11:	[0s / 2s],		train_loss: 0.1135,	val_loss: 0.1079
12:	[0s / 2s],		train_loss: 0.1135,	val_loss: 0.1076
13:	[0s / 2s],		train_loss: 0.1125,	val_loss: 0.1079
14:	[0s / 2s],		train_loss: 0.1127,	val_loss: 0.1079
15:	[0s / 3s],		train_loss: 0.1121,	val_loss: 0.1078
16:	[0s / 3s],		train_loss: 0.1123,	val_loss: 0.1078
17:	[0s / 3s],		train_loss: 0.1117,	val_loss: 0.1080
18:	[0s / 3s],		train_loss: 0.1123,	val_loss: 0.1078
19:

In [25]:
# get probabilites
probs_train = torch.sigmoid(model.predict(x_train))
probs_test = torch.sigmoid(model.predict(x_test))

print(f'log-loss is: {log_loss(y_train, probs_train):.4f}')
print(f'log-loss os: {log_loss(y_test, probs_test):.4f}')
print(f'BS is: {brier_score_loss(y_train, probs_train):.4f}')
print(f'BS os: {brier_score_loss(y_test, probs_test):.4f}')

log-loss is: 0.1099
log-loss os: 0.1092
BS is: 0.0292
BS os: 0.0292


In [26]:
# DeepSC2B

In [27]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)  
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ['PYTHONHASHSEED'] = str(SEED)

# define the network
in_features = x_train.shape[1]
net = tt.practical.MLPVanilla(in_features, [64,32], 1, activation=nn.ReLU, dropout=0.5)

# model and optimizer
loss_fn = nn.BCEWithLogitsLoss()
model = tt.Model(net, loss_fn, tt.optim.Adam)  
model.optimizer.set_lr(0.005)  

# training setup     
batch_size = 200
epochs = 60
# early stopping
checkpoint_path = 'best_model.pt'
if os.path.exists(checkpoint_path):
    os.remove(checkpoint_path)
callbacks = [EarlyStopping(patience=20, file_path=checkpoint_path)]

# fit
log = model.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, val_data=(x_test, y_test), shuffle=True,
                callbacks=callbacks, verbose=True)

0:	[0s / 0s],		train_loss: 0.2204,	val_loss: 0.1108
1:	[0s / 1s],		train_loss: 0.1249,	val_loss: 0.1099
2:	[0s / 2s],		train_loss: 0.1215,	val_loss: 0.1089
3:	[0s / 3s],		train_loss: 0.1191,	val_loss: 0.1084
4:	[0s / 4s],		train_loss: 0.1193,	val_loss: 0.1087
5:	[0s / 5s],		train_loss: 0.1181,	val_loss: 0.1088
6:	[0s / 5s],		train_loss: 0.1174,	val_loss: 0.1093
7:	[0s / 6s],		train_loss: 0.1178,	val_loss: 0.1088
8:	[0s / 7s],		train_loss: 0.1176,	val_loss: 0.1089
9:	[0s / 8s],		train_loss: 0.1171,	val_loss: 0.1078
10:	[0s / 8s],		train_loss: 0.1148,	val_loss: 0.1080
11:	[0s / 9s],		train_loss: 0.1154,	val_loss: 0.1077
12:	[0s / 10s],		train_loss: 0.1158,	val_loss: 0.1086
13:	[0s / 11s],		train_loss: 0.1152,	val_loss: 0.1081
14:	[0s / 11s],		train_loss: 0.1150,	val_loss: 0.1080
15:	[0s / 12s],		train_loss: 0.1146,	val_loss: 0.1088
16:	[0s / 13s],		train_loss: 0.1142,	val_loss: 0.1089
17:	[0s / 14s],		train_loss: 0.1140,	val_loss: 0.1085
18:	[1s / 15s],		train_loss: 0.1140,	val_loss: 0.1

In [28]:
# get probabilites
probs_train = torch.sigmoid(model.predict(x_train))
probs_test = torch.sigmoid(model.predict(x_test))

print(f'log-loss is: {log_loss(y_train, probs_train):.4f}')
print(f'log-loss os: {log_loss(y_test, probs_test):.4f}')
print(f'BS is: {brier_score_loss(y_train, probs_train):.4f}')
print(f'BS os: {brier_score_loss(y_test, probs_test):.4f}')

log-loss is: 0.1071
log-loss os: 0.1093
BS is: 0.0290
BS os: 0.0292


In [29]:
# DeepSC3B

In [30]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)  
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ['PYTHONHASHSEED'] = str(SEED)

# define the network
in_features = x_train.shape[1]
net = tt.practical.MLPVanilla(in_features, [16], 1, activation=nn.ReLU, dropout=0.5)

# model and optimizer
loss_fn = nn.BCEWithLogitsLoss()
model = tt.Model(net, loss_fn, tt.optim.Adam)  
model.optimizer.set_lr(0.005)  

# training setup   
batch_size = 800
epochs = 30
# early stopping
checkpoint_path = 'best_model.pt'
if os.path.exists(checkpoint_path):
    os.remove(checkpoint_path)
callbacks = [EarlyStopping(patience=20, file_path=checkpoint_path)]

# fit
log = model.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, val_data=(x_test, y_test), shuffle=True,
                callbacks=callbacks, verbose=True)

0:	[0s / 0s],		train_loss: 0.4472,	val_loss: 0.1464
1:	[0s / 0s],		train_loss: 0.1419,	val_loss: 0.1109
2:	[0s / 0s],		train_loss: 0.1270,	val_loss: 0.1097
3:	[0s / 0s],		train_loss: 0.1238,	val_loss: 0.1090
4:	[0s / 0s],		train_loss: 0.1207,	val_loss: 0.1084
5:	[0s / 1s],		train_loss: 0.1198,	val_loss: 0.1086
6:	[0s / 1s],		train_loss: 0.1202,	val_loss: 0.1082
7:	[0s / 1s],		train_loss: 0.1192,	val_loss: 0.1084
8:	[0s / 1s],		train_loss: 0.1192,	val_loss: 0.1080
9:	[0s / 1s],		train_loss: 0.1194,	val_loss: 0.1082
10:	[0s / 2s],		train_loss: 0.1183,	val_loss: 0.1080
11:	[0s / 2s],		train_loss: 0.1188,	val_loss: 0.1083
12:	[0s / 2s],		train_loss: 0.1188,	val_loss: 0.1081
13:	[0s / 2s],		train_loss: 0.1172,	val_loss: 0.1081
14:	[0s / 2s],		train_loss: 0.1175,	val_loss: 0.1081
15:	[0s / 2s],		train_loss: 0.1175,	val_loss: 0.1081
16:	[0s / 3s],		train_loss: 0.1179,	val_loss: 0.1079
17:	[0s / 3s],		train_loss: 0.1180,	val_loss: 0.1082
18:	[0s / 3s],		train_loss: 0.1180,	val_loss: 0.1082
19:

In [31]:
# get probabilites
probs_train = torch.sigmoid(model.predict(x_train))
probs_test = torch.sigmoid(model.predict(x_test))

print(f'log-loss is: {log_loss(y_train, probs_train):.4f}')
print(f'log-loss os: {log_loss(y_test, probs_test):.4f}')
print(f'BS is: {brier_score_loss(y_train, probs_train):.4f}')
print(f'BS os: {brier_score_loss(y_test, probs_test):.4f}')

log-loss is: 0.1110
log-loss os: 0.1094
BS is: 0.0294
BS os: 0.0291
